# 02 — Fixed-test architecture comparison

Evaluate the four completed end-to-end systems on the fixed 15-knee test set. Results are aggregated to subject level before the primary comparison so bilateral healthy knees do not receive twice the weight of fractured subjects.

Set `RUN_EVALUATION = True` only after notebook 01 has produced all four `best_model.pth` checkpoints.

In [ ]:
import contextlib
import math
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from scipy.ndimage import binary_erosion, distance_transform_edt
from torch.utils.data import DataLoader, Dataset

SEED = 42
RUN_EVALUATION = False
ARMS = [
    "plain_unet_style",
    "plain_prelu_style",
    "residual_relu_style",
    "residual_vnet_style",
]
ARM_FACTORS = {
    "plain_unet_style": {"activation": "relu", "residual": False, "label": "U (ReLU, plain)"},
    "plain_prelu_style": {"activation": "prelu", "residual": False, "label": "PReLU + plain"},
    "residual_relu_style": {"activation": "relu", "residual": True, "label": "ReLU + residual"},
    "residual_vnet_style": {"activation": "prelu", "residual": True, "label": "V (PReLU, residual)"},
}
BONES = ["femur", "tibia", "patella", "fibula"]
FEATURE_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attention", "attention"]
IMAGE_SIZE = 256
LOGIT_SIZE = 128
SPACING_MM = (0.78125, 0.78125, 0.78125)
NUM_WORKERS = 2
N_BOOTSTRAP = 10_000
USE_AMP = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: full test inference is intended for the HPC CUDA environment.")
print({"device": str(DEVICE), "run_evaluation": RUN_EVALUATION})

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "reports" / "manifests" / "quantitative_manifest_v1.csv").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current working directory.")


ROOT = find_project_root(Path.cwd())
MANIFEST_PATH = ROOT / "reports" / "manifests" / "quantitative_manifest_v1.csv"
MODEL_ROOT = ROOT / "models" / "joint_simclr_test"
SIMCLR_PATH = MODEL_ROOT / "simclr" / "simclr_encoder.pth"
REPORT_DIR = ROOT / "reports" / "joint_simclr_test"


def load_test_rows():
    rows = pd.read_csv(MANIFEST_PATH, dtype={"test_fold": "Int64"})
    rows = rows.loc[rows.status.eq("ready")].copy()
    if len(rows) != 71:
        raise RuntimeError(f"Expected 71 ready knees, found {len(rows)}.")
    rows["test_fold"] = rows.test_fold.astype(int)
    train_rows = rows.loc[rows.test_fold.isin([2, 3, 4])]
    validation_rows = rows.loc[rows.test_fold.eq(1)]
    test_rows = rows.loc[rows.test_fold.eq(0)].copy()
    if (len(train_rows), len(validation_rows), len(test_rows)) != (42, 14, 15):
        raise RuntimeError("The fixed 42/14/15 split is not present in the certified manifest.")
    subject_sets = [set(frame.subject_id) for frame in (train_rows, validation_rows, test_rows)]
    if subject_sets[0] & subject_sets[1] or subject_sets[0] & subject_sets[2] or subject_sets[1] & subject_sets[2]:
        raise RuntimeError("Subject overlap detected between fixed split roles.")
    if test_rows.subject_id.nunique() != 9 or test_rows.dataset.value_counts().to_dict() != {"VSD": 12, "Ruikar": 3}:
        raise RuntimeError("Expected 15 test knees from nine subjects: 12 VSD and three Ruikar.")
    return test_rows.sort_values("sample_id").reset_index(drop=True)


test_rows = load_test_rows()
display(pd.DataFrame([{
    "test_knees": len(test_rows),
    "test_subjects": test_rows.subject_id.nunique(),
    "healthy_knees": int(test_rows.dataset.eq("VSD").sum()),
    "fractured_knees": int(test_rows.dataset.eq("Ruikar").sum()),
}]))

In [ ]:
def read_drr(path: Path) -> np.ndarray:
    array = np.load(path).astype(np.float32)
    if array.shape != (IMAGE_SIZE, IMAGE_SIZE) or not np.isfinite(array).all():
        raise ValueError(f"Invalid DRR: {path}")
    return np.clip(array, 0.0, 1.0)


def load_target(row) -> torch.Tensor:
    arrays = []
    target_dir = ROOT / row.target_path
    for bone in BONES:
        image = nib.load(str(target_dir / f"{row.sample_id}_{bone}.nii.gz"))
        if image.shape != (IMAGE_SIZE, IMAGE_SIZE, IMAGE_SIZE) or tuple(nib.aff2axcodes(image.affine)) != ("L", "P", "S"):
            raise ValueError(f"Target geometry mismatch: {row.sample_id}/{bone}")
        array = np.asarray(image.dataobj, dtype=np.float32)
        if array.sum() <= 0 or not set(np.unique(array).tolist()).issubset({0.0, 1.0}):
            raise ValueError(f"Target must be binary and non-empty: {row.sample_id}/{bone}")
        arrays.append(array)
    return torch.from_numpy(np.stack(arrays).astype(np.float32))


class TestDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        return {
            "ap": torch.from_numpy(read_drr(ROOT / row.ap_drr_path)).unsqueeze(0),
            "lat": torch.from_numpy(read_drr(ROOT / row.lat_drr_path)).unsqueeze(0),
            "target": load_target(row),
            "sample_id": str(row.sample_id),
            "subject_id": str(row.subject_id),
            "dataset": str(row.dataset),
        }


def binary_metrics(prediction, target, spacing=SPACING_MM):
    prediction = np.asarray(prediction, dtype=bool)
    target = np.asarray(target, dtype=bool)
    intersection = np.logical_and(prediction, target).sum(dtype=np.float64)
    denominator = prediction.sum(dtype=np.float64) + target.sum(dtype=np.float64)
    dice = float(2.0 * intersection / denominator) if denominator else 1.0
    if not prediction.any() and not target.any():
        return dice, 0.0, 0.0
    if not prediction.any() or not target.any():
        penalty = float(np.linalg.norm((np.asarray(prediction.shape) - 1) * np.asarray(spacing)))
        return dice, penalty, penalty
    prediction_surface = prediction ^ binary_erosion(prediction)
    target_surface = target ^ binary_erosion(target)
    distance_to_target = distance_transform_edt(~target_surface, sampling=spacing)[prediction_surface]
    distance_to_prediction = distance_transform_edt(~prediction_surface, sampling=spacing)[target_surface]
    distances = np.concatenate([distance_to_target, distance_to_prediction])
    return dice, float(distances.mean()), float(np.percentile(distances, 95))


def metric_unit_tests():
    target = np.zeros((9, 9, 9), dtype=bool)
    target[2:7, 2:7, 2:7] = True
    identical = binary_metrics(target, target, spacing=(1, 1, 1))
    empty = binary_metrics(np.zeros_like(target), target, spacing=(1, 1, 1))
    if not np.allclose(identical, (1.0, 0.0, 0.0)):
        raise AssertionError(f"Identical-mask metric test failed: {identical}")
    if empty[0] != 0.0 or not np.isfinite(empty[1:]).all():
        raise AssertionError(f"Empty-prediction metric test failed: {empty}")
    return {"identical": identical, "empty_prediction": empty}


print("metric tests:", metric_unit_tests())

In [ ]:
def make_activation(kind: str, channels: int):
    if kind == "relu":
        return nn.ReLU(inplace=True)
    if kind == "prelu":
        return nn.PReLU(channels)
    raise ValueError(f"Unknown activation: {kind}")


class CrossAttention(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.query = nn.Linear(channels, channels)
        self.key = nn.Linear(channels, channels)
        self.value = nn.Linear(channels, channels)
        self.scale = channels ** -0.5

    def forward(self, query_map, context_map):
        batch, channels, height, width = query_map.shape
        query = query_map.flatten(2).transpose(1, 2)
        context = context_map.flatten(2).transpose(1, 2)
        weights = torch.softmax(self.query(query) @ self.key(context).transpose(-2, -1) * self.scale, dim=-1)
        return (weights @ self.value(context) + query).transpose(1, 2).reshape(batch, channels, height, width)


class LocalFusion(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.mix = nn.Conv2d(2 * channels, channels, 3, padding=1)

    def forward(self, query_map, context_map):
        return self.mix(torch.cat([query_map, context_map], dim=1)) + query_map


class DecoderBlock(nn.Module):
    def __init__(self, input_channels: int, output_channels: int, activation: str, residual: bool):
        super().__init__()
        self.residual = residual
        self.projection = None
        if residual:
            self.projection = nn.Conv3d(input_channels, output_channels, 1) if input_channels != output_channels else nn.Identity()
        self.conv1 = nn.Conv3d(input_channels, output_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, output_channels)
        self.act1 = make_activation(activation, output_channels)
        self.conv2 = nn.Conv3d(output_channels, output_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, output_channels)
        self.act2 = make_activation(activation, output_channels)

    def forward(self, value):
        residual = self.projection(value) if self.residual else None
        value = self.act1(self.norm1(self.conv1(value)))
        value = self.norm2(self.conv2(value))
        if residual is not None:
            value = value + residual
        return self.act2(value)


class MatchedDecoder3D(nn.Module):
    def __init__(self, arm: str):
        super().__init__()
        factors = ARM_FACTORS[arm]
        activation, residual = factors["activation"], factors["residual"]
        c0, c1, c2, c3 = FEATURE_CHANNELS
        self.up3 = nn.ConvTranspose3d(c3, c2, 2, 2)
        self.dec3 = DecoderBlock(c2 + c2, c2, activation, residual)
        self.up2 = nn.ConvTranspose3d(c2, c1, 2, 2)
        self.dec2 = DecoderBlock(c1 + c1, c1, activation, residual)
        self.up1 = nn.ConvTranspose3d(c1, c0, 2, 2)
        self.dec1 = DecoderBlock(c0 + c0, c0, activation, residual)
        self.output = nn.Conv3d(c0, len(BONES), 1)

    def forward(self, levels):
        level0, level1, level2, level3 = levels
        value = self.dec3(torch.cat([self.up3(level3), level2], dim=1))
        value = self.dec2(torch.cat([self.up2(value), level1], dim=1))
        value = self.dec1(torch.cat([self.up1(value), level0], dim=1))
        value = F.interpolate(value, size=(LOGIT_SIZE,) * 3, mode="trilinear", align_corners=False)
        logits = self.output(value)
        return F.interpolate(logits, size=(IMAGE_SIZE,) * 3, mode="trilinear", align_corners=False)


class JointReconstructionModel(nn.Module):
    def __init__(self, arm: str, simclr_checkpoint):
        super().__init__()
        self.encoder = timm.create_model(simclr_checkpoint["model_name"], pretrained=False, num_classes=0)
        self.encoder.load_state_dict(simclr_checkpoint["encoder_state"], strict=True)
        self.normalization = simclr_checkpoint["normalization"]
        source_channels = [96, 192, 384, 768]
        self.fusion = nn.ModuleList([
            CrossAttention(channels) if kind == "attention" else LocalFusion(channels)
            for channels, kind in zip(source_channels, FUSION_TYPES)
        ])
        self.project_2d = nn.ModuleList([nn.Conv2d(source, target, 1) for source, target in zip(source_channels, FEATURE_CHANNELS)])
        self.fuse_3d = nn.ModuleList([nn.Conv3d(2 * channels, channels, 3, padding=1) for channels in FEATURE_CHANNELS])
        self.decoder = MatchedDecoder3D(arm)

    def normalize(self, raw):
        image = raw.repeat(1, 3, 1, 1)
        mean = torch.tensor(self.normalization["mean"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        std = torch.tensor(self.normalization["std"], device=image.device, dtype=image.dtype).view(1, 3, 1, 1)
        return (image - mean) / std

    def encode(self, raw):
        _, levels = self.encoder.forward_intermediates(self.normalize(raw), indices=(0, 1, 2, 3))
        return tuple(levels)

    @staticmethod
    def lift(ap_feature, lat_feature, projection, fusion_3d):
        ap = projection(ap_feature)
        lat = projection(lat_feature).flip(3)
        batch, channels, size, _ = ap.shape
        ap_cube = ap.permute(0, 1, 3, 2).unsqueeze(3).expand(batch, channels, size, size, size)
        lat_cube = lat.permute(0, 1, 3, 2).unsqueeze(2).expand(batch, channels, size, size, size)
        return fusion_3d(torch.cat([ap_cube, lat_cube], dim=1))

    def forward(self, ap_raw, lat_raw):
        ap_levels, lat_levels = self.encode(ap_raw), self.encode(lat_raw)
        output = []
        for ap, lat, fusion, projection, fusion_3d in zip(ap_levels, lat_levels, self.fusion, self.project_2d, self.fuse_3d):
            output.append(self.lift(fusion(ap, lat), fusion(lat, ap), projection, fusion_3d))
        return self.decoder(output)

In [ ]:
def amp_context():
    if not (USE_AMP and DEVICE.type == "cuda"):
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast("cuda", dtype=dtype)


def bootstrap_interval(values, rng, n_bootstrap=N_BOOTSTRAP):
    values = np.asarray(values, dtype=np.float64)
    estimates = np.empty(n_bootstrap, dtype=np.float64)
    for index in range(n_bootstrap):
        estimates[index] = rng.choice(values, size=len(values), replace=True).mean()
    return float(values.mean()), float(np.percentile(estimates, 2.5)), float(np.percentile(estimates, 97.5))


def factorial_subject_effects(subject_frame, metric):
    wide = subject_frame.pivot(index="subject_id", columns="arm", values=metric)
    required = set(ARMS)
    if set(wide.columns) != required or wide.isna().any().any():
        raise RuntimeError(f"Incomplete matched subject table for {metric}.")
    u = wide["plain_unet_style"].to_numpy()
    p = wide["plain_prelu_style"].to_numpy()
    r = wide["residual_relu_style"].to_numpy()
    v = wide["residual_vnet_style"].to_numpy()
    return {
        "residual_minus_plain": 0.5 * (r + v) - 0.5 * (u + p),
        "prelu_minus_relu": 0.5 * (p + v) - 0.5 * (u + r),
        "residual_by_activation_interaction": v - r - p + u,
        "vnet_style_minus_unet_style": v - u,
    }


def save_comparison_plots(subject_frame):
    labels = [ARM_FACTORS[arm]["label"] for arm in ARMS]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    dice_values = [subject_frame.loc[subject_frame.arm.eq(arm), "dice"].to_numpy() for arm in ARMS]
    assd_values = [subject_frame.loc[subject_frame.arm.eq(arm), "assd_mm"].to_numpy() for arm in ARMS]
    axes[0].boxplot(dice_values, tick_labels=labels, showmeans=True)
    axes[1].boxplot(assd_values, tick_labels=labels, showmeans=True)
    axes[0].set(title="Subject-macro Dice", ylabel="Dice")
    axes[1].set(title="Subject-macro ASSD", ylabel="ASSD (mm)")
    for axis in axes:
        axis.tick_params(axis="x", rotation=25)
        axis.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "architecture_boxplots.png", dpi=180, bbox_inches="tight")
    plt.show()

    wide = subject_frame.pivot(index="subject_id", columns="arm", values="dice")
    fig, axis = plt.subplots(figsize=(9, 5))
    for _, row in wide.iterrows():
        axis.plot(range(len(ARMS)), row[ARMS], marker="o", alpha=0.55)
    axis.set_xticks(range(len(ARMS)), labels, rotation=25, ha="right")
    axis.set(title="Paired test-subject Dice", ylabel="Subject-macro Dice")
    axis.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "paired_subject_dice.png", dpi=180, bbox_inches="tight")
    plt.show()


def save_qa_montages(qa_slices):
    for sample_id, content in qa_slices.items():
        fig, axes = plt.subplots(1, 1 + len(ARMS), figsize=(18, 4))
        axes[0].imshow(content["target"], cmap="gray")
        axes[0].set_title(f"{sample_id}\nTarget union")
        for axis, arm in zip(axes[1:], ARMS):
            axis.imshow(content[arm], cmap="gray")
            axis.set_title(ARM_FACTORS[arm]["label"])
        for axis in axes:
            axis.axis("off")
        fig.tight_layout()
        fig.savefig(REPORT_DIR / f"qa_{sample_id}.png", dpi=180, bbox_inches="tight")
        plt.show()


def run_evaluation():
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    if not SIMCLR_PATH.is_file():
        raise FileNotFoundError(f"Missing SimCLR checkpoint: {SIMCLR_PATH}")
    checkpoint_paths = {arm: MODEL_ROOT / arm / "best_model.pth" for arm in ARMS}
    missing = [str(path) for path in checkpoint_paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Complete notebook 01 before testing. Missing checkpoints: {missing}")

    simclr_checkpoint = torch.load(SIMCLR_PATH, map_location="cpu", weights_only=False)
    loader = DataLoader(TestDataset(test_rows), batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0)
    selected_qa = {
        test_rows.loc[test_rows.dataset.eq("VSD"), "sample_id"].iloc[0],
        test_rows.loc[test_rows.dataset.eq("Ruikar"), "sample_id"].iloc[0],
    }
    qa_slices, metric_records = {}, []

    for arm in ARMS:
        model = JointReconstructionModel(arm, simclr_checkpoint)
        best = torch.load(checkpoint_paths[arm], map_location="cpu", weights_only=False)
        if best["arm"] != arm:
            raise RuntimeError(f"Checkpoint arm mismatch: expected {arm}, found {best['arm']}.")
        model.load_state_dict(best["model_state"], strict=True)
        model.to(DEVICE).eval()
        with torch.no_grad():
            for batch in loader:
                ap = batch["ap"].to(DEVICE, non_blocking=True)
                lat = batch["lat"].to(DEVICE, non_blocking=True)
                with amp_context():
                    logits = model(ap, lat)
                prediction = (torch.sigmoid(logits.float()) > 0.5)[0].cpu().numpy()
                target = batch["target"][0].numpy() > 0.5
                sample_id = batch["sample_id"][0]
                subject_id = batch["subject_id"][0]
                dataset = batch["dataset"][0]
                cohort = "fractured" if dataset == "Ruikar" else "healthy"
                for bone_index, bone in enumerate(BONES):
                    dice, assd, hd95 = binary_metrics(prediction[bone_index], target[bone_index])
                    metric_records.append({
                        "arm": arm,
                        "arm_label": ARM_FACTORS[arm]["label"],
                        "sample_id": sample_id,
                        "subject_id": subject_id,
                        "dataset": dataset,
                        "cohort": cohort,
                        "bone": bone,
                        "dice": dice,
                        "assd_mm": assd,
                        "hd95_mm": hd95,
                    })
                if sample_id in selected_qa:
                    target_union = target.any(axis=0)
                    slice_index = int(np.argmax(target_union.sum(axis=(0, 1))))
                    content = qa_slices.setdefault(sample_id, {"target": target_union[:, :, slice_index]})
                    content[arm] = prediction.any(axis=0)[:, :, slice_index]
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metrics = pd.DataFrame(metric_records)
    if len(metrics) != len(ARMS) * 15 * len(BONES) or not np.isfinite(metrics[["dice", "assd_mm", "hd95_mm"]].to_numpy()).all():
        raise RuntimeError("Test metrics are incomplete or non-finite.")
    metrics.to_csv(REPORT_DIR / "test_metrics.csv", index=False)

    subject_frame = metrics.groupby(["arm", "arm_label", "subject_id"], as_index=False)[["dice", "assd_mm", "hd95_mm"]].mean()
    rng = np.random.default_rng(SEED)
    summary_rows = []
    for arm in ARMS:
        arm_subjects = subject_frame.loc[subject_frame.arm.eq(arm)]
        for metric in ("dice", "assd_mm", "hd95_mm"):
            estimate, lower, upper = bootstrap_interval(arm_subjects[metric], rng)
            summary_rows.append({"type": "arm", "name": arm, "metric": metric, "estimate": estimate, "ci_lower": lower, "ci_upper": upper})

    for metric in ("dice", "assd_mm"):
        for effect_name, values in factorial_subject_effects(subject_frame, metric).items():
            estimate, lower, upper = bootstrap_interval(values, rng)
            summary_rows.append({"type": "factorial_effect", "name": effect_name, "metric": metric, "estimate": estimate, "ci_lower": lower, "ci_upper": upper})

    comparison_summary = pd.DataFrame(summary_rows)
    comparison_summary.to_csv(REPORT_DIR / "comparison_summary.csv", index=False)
    per_bone = metrics.groupby(["arm", "bone"], as_index=False)[["dice", "assd_mm", "hd95_mm"]].mean()
    per_cohort_subject = metrics.groupby(["arm", "cohort", "subject_id"], as_index=False)[["dice", "assd_mm", "hd95_mm"]].mean()
    per_cohort = per_cohort_subject.groupby(["arm", "cohort"], as_index=False)[["dice", "assd_mm", "hd95_mm"]].mean()
    per_bone.to_csv(REPORT_DIR / "per_bone_summary.csv", index=False)
    per_cohort.to_csv(REPORT_DIR / "per_cohort_summary.csv", index=False)

    display(comparison_summary)
    display(per_bone)
    display(per_cohort)
    save_comparison_plots(subject_frame)
    save_qa_montages(qa_slices)
    print({"report_directory": str(REPORT_DIR), "metric_rows": len(metrics), "subjects": subject_frame.subject_id.nunique()})
    return metrics, comparison_summary, per_bone, per_cohort


if RUN_EVALUATION:
    test_metrics, comparison_summary, per_bone_summary, per_cohort_summary = run_evaluation()
else:
    print("Definitions loaded. Set RUN_EVALUATION=True after all four best_model.pth checkpoints exist.")